### Import packages

In [ ]:
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import adjusted_rand_score
import numpy as np
np.random.seed(42)

### Load files needed
You will need files ending in logCPM.csv.gz, two HVGSs.txt files and the metadata.bminty.txt.gz

In [ ]:
matrix = pd.read_csv(
    "/pbmc.bead.enriched.sample.zheng.filtered.logCPM.csv.gz",
    index_col=0
)
matrix

In [ ]:
matrix = matrix.T
matrix

In [ ]:
metadata=pd.read_csv("/pbmc.bead.enriched.sample.zheng.metadata.bminty.txt.gz",index_col=0,sep='|')
metadata

In [ ]:
adata = sc.AnnData(matrix)
adata

In [ ]:
adata.obs['cell_type']=metadata['cell_type']
adata.obs['cell_type']

In [ ]:
# Load HVGs
hvgs_mvp = pd.read_csv("/Seurat_mvp_HVGs.txt",header=None)[0].str.replace(r"^\d+\.", "", regex=True)
hvgs_vst = pd.read_csv("/Seurat_vst_HVGs.txt",header=None)[0].str.replace(r"^\d+\.", "", regex=True)

In [ ]:
adata_mvp = adata[:, hvgs_mvp].copy()
adata_vst = adata[:, hvgs_vst].copy()

### PCA - Clustering

In [ ]:
sc.tl.pca(adata_mvp)
sc.tl.pca(adata_vst)

In [ ]:
sc.pl.pca(adata_mvp, color="cell_type")

In [ ]:
sc.pl.pca(adata_vst, color="cell_type")

In [ ]:
sc.pl.pca_variance_ratio(adata_mvp, n_pcs=50, log=True)

In [ ]:
sc.pl.pca_variance_ratio(adata_vst, n_pcs=50, log=True)

Here we will just use one of the objects to just check a little bit the clustering result.

In [ ]:
N_PCS = 20
sc.pp.neighbors(adata_vst, n_pcs=N_PCS)
sc.tl.umap(adata_vst)
sc.tl.leiden(adata_vst, resolution=0.5, key_added="leiden_r_0.5",flavor="igraph")

In [ ]:
sc.pl.umap(adata_vst, color="leiden_r_0.5", legend_loc="on data")

In [ ]:
sc.tl.tsne(adata_vst)
sc.pl.tsne(adata_vst, color="leiden_r_0.5", legend_loc="on data")

### Ok let's observe the effect of some parameters for both objects now

In [ ]:
k_values = [5, 15, 30]
resolutions = [0.3, 0.5, 0.8, 1]

In [ ]:
for k in k_values:
    sc.pp.neighbors(
        adata_mvp,
        n_neighbors=k,
        n_pcs=N_PCS,
        key_added=f"neighbors_k{k}"
    )

    sc.tl.umap(
        adata_mvp,
        neighbors_key=f"neighbors_k{k}",
        key_added=f"X_umap_k{k}_"
    )

    for res in resolutions:
        cluster_key = f"leiden_k{k}_r_{res}"
        sc.tl.leiden(
            adata_mvp,
            resolution=res,
            neighbors_key=f"neighbors_k{k}",
            key_added=cluster_key,
            flavor="igraph"
        )

        sc.pl.embedding(
        adata_mvp,
        basis=f"umap_k{k}_",
        color=cluster_key,
        title=f"k={k}, resolution={res}",
        legend_loc="on data"
        )

In [ ]:
for k in k_values:
    sc.pp.neighbors(
        adata_vst,
        n_neighbors=k,
        n_pcs=N_PCS,
        key_added=f"neighbors_k{k}"
    )

    sc.tl.umap(
        adata_vst,
        neighbors_key=f"neighbors_k{k}",
        key_added=f"X_umap_k{k}_"
    )

    for res in resolutions:
        cluster_key = f"leiden_k{k}_r_{res}"
        sc.tl.leiden(
            adata_vst,
            resolution=res,
            neighbors_key=f"neighbors_k{k}",
            key_added=cluster_key,
            flavor="igraph"
        )

        sc.pl.embedding(
        adata_vst,
        basis=f"umap_k{k}_",
        color=cluster_key,
        title=f"k={k}, resolution={res}",
        legend_loc="on data"
        )

Format of anndata is clear?

In [ ]:
adata_vst

In [ ]:
adata_vst.obs

### Use ARI -external clustering evaluation metric- to identify the best parameters.
Example ARI dataframe
| k_neighbors | resolution | clusters | ARI     | Method_hvgs |
|-------------|------------|----------|----------|--------------|
| 5           | 0.3        | 8        | 0.549030 | Seurat_vst   |
| 5           | 0.3        | 9        | 0.549072 | Seurat_mvp   |

In [ ]:
reference_key = "cell_type" 
ari_results = []

for k in k_values:
    for res in resolutions:
        cluster_key = f"leiden_k{k}_r_{res}"

        for method, ad in zip(
            ["Seurat_vst", "Seurat_mvp"],
            [adata_vst, adata_mvp]):

            ari = adjusted_rand_score( 
                ad.obs[reference_key],
                ad.obs[cluster_key]
            )
   
            ari_results.append({
                "k_neighbors": k,
                "resolution": res,
                "clusters": ad.obs[cluster_key].nunique(),
                "ARI": ari,
                "Method_hvgs":method
            })

ari_df = pd.DataFrame(ari_results)
ari_df 

In [ ]:
best_combinations = (ari_df.sort_values("ARI", ascending=False).groupby("Method_hvgs").head(1))
best_combinations

In [ ]:
sns.lineplot(
    data=ari_df,
    x="resolution",
    y="ARI",
    hue="k_neighbors",
    style="Method_hvgs",
    palette=["blue", "orange", "green"],
    marker="o"
)

plt.title("Effect of k and Leiden resolution on ARI")
plt.show()

In [ ]:
adata_vst = adata[:, hvgs_vst].copy()
sc.tl.pca(adata_vst)
best_k = 15
best_res = 0.8
sc.pp.neighbors(adata_vst, n_pcs=N_PCS, n_neighbors=best_k)
sc.tl.umap(adata_vst)
sc.tl.leiden(adata_vst, resolution=best_res,flavor="igraph")
adata_vst.write("/adata_vst_final.h5ad")

### That's all from this part! Thank you!